# Task 1 — Rating prediction (latent factor model)

Explore and pick hyperparameters here; copy the winner into `src/assignment_1/tasks/task1.py`.

HW3 target: public MSE ≤ 0.7280 (baseline 0.7429).

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools
from math import prod

import pandas as pd
from surprise import BaselineOnly
from tqdm.auto import tqdm

from assignment_1.modules.models.rating import fit_svd, predict_svd, to_surprise_trainset
from assignment_1.modules.util import calc_mse, load_csv, random_split

DATA_DIR = "../data"  # notebook runs from notebooks/

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/workspaces/cse-258-assignment-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Look at the data
Check the rating range: Surprise clips predictions to it, and the baseline assumed 1–5.

In [25]:
train = load_csv("tasks_1_2_train.csv", DATA_DIR)
test = load_csv("task_1_test.csv", DATA_DIR)

print(
    f"{len(train):,} ratings | {train.user.nunique():,} users | {train.item.nunique():,} items"
)
print(
    f"test users unseen in train: {(~test.user.isin(train.user)).mean():.1%}, items: {(~test.item.isin(train.item)).mean():.1%}"
)

train.rating.value_counts().sort_index()

1,861,643 ratings | 30,909 users | 51,007 items
test users unseen in train: 0.0%, items: 0.0%


rating
0     27626
1     36443
2     97748
3    314624
4    641182
5    744020
Name: count, dtype: int64

## Validation split + baseline yardstick
Same split for every model, so differences come from the model, not the split. `rating_scale` comes from the *full* training data so validation-split models clip the same way the final model does.

In [26]:
tr, val = random_split(train, val_frac=0.25, seed=0)
SCALE = (train.rating.min(), train.rating.max())


def val_mse(**params):
    model = fit_svd(tr, rating_scale=SCALE, **params)
    return calc_mse(val.rating, predict_svd(model, val.user, val.item))


base = val_mse(n_factors=1, n_epochs=50)
print(f"baseline val MSE = {base:.4f}  ->  2% target ≈ {base * 0.98:.4f}")

baseline val MSE = 0.7863  ->  2% target ≈ 0.7705


## Step 1: sweep regularization only (λ = `reg_all`)
If the best value is at the edge of the range, extend the range.

In [31]:
grid = {"n_factors": [1, 10, 50], "reg_all": [0.1, 0.3, 0.5], "n_epochs": [50]}

rows = []
for values in tqdm(
    itertools.product(*grid.values()), total=prod(map(len, grid.values()))
):
    params = dict(zip(grid, values))
    s = val_mse(**params)
    rows.append({**params, "val_mse": s, "vs_base_%": (1 - s / base) * 100})

pd.DataFrame(rows).sort_values("val_mse")

  0%|          | 0/9 [00:00<?, ?it/s]

100%|██████████| 9/9 [05:17<00:00, 35.25s/it]


,n_factors,reg_all,n_epochs,val_mse,vs_base_%
0,1,0.1,50,0.779227,0.893713
3,10,0.1,50,0.780010,0.794045
6,50,0.1,50,0.780975,0.671293
1,1,0.3,50,0.796082,-1.249984
4,10,0.3,50,0.796128,-1.255924
7,50,0.3,50,0.796306,-1.278558
2,1,0.5,50,0.818003,-4.038029
5,10,0.5,50,0.818026,-4.041043
8,50,0.5,50,0.818105,-4.051015


## Next
- Sweep `n_factors`, `lr_all`, `n_epochs` around the best λ.
- Copy the winner into `TASK1_SVD_PARAMS` in `tasks/task1.py`, then `make predict` (or `make submit`).

In [28]:
def val_mse_bias(reg_u, reg_i, n_epochs=10):
    algo = BaselineOnly(
        bsl_options={
            "method": "als",
            "reg_u": reg_u,
            "reg_i": reg_i,
            "n_epochs": n_epochs,
        },
        verbose=False,
    )
    algo.fit(to_surprise_trainset(tr, SCALE))
    preds = [algo.predict(u, i).est for u, i in zip(val.user, val.item)]

    return calc_mse(val.rating, preds)


grid = {"reg_u": [0.1, 0.3, 0.5, 1], "reg_i": [3, 5, 7], "n_epochs": [10, 25]}

rows = []
for ru, ri, ep in tqdm(
    itertools.product(*grid.values()), total=prod(map(len, grid.values()))
):
    s = val_mse_bias(ru, ri, n_epochs=ep)
    rows.append(
        {
            "reg_u": ru,
            "reg_i": ri,
            "n_epochs": ep,
            "val_mse": s,
            "vs_base_%": (1 - s / base) * 100,
        }
    )

results = pd.DataFrame(rows).sort_values("val_mse")
results.head(10)

100%|██████████| 24/24 [04:43<00:00, 11.81s/it]


,reg_u,reg_i,n_epochs,val_mse,vs_base_%
21,1.0,5,25,0.769797,2.093000
20,1.0,5,10,0.769949,2.073670
15,0.5,5,25,0.769963,2.071918
9,0.3,5,25,0.770153,2.047809
14,0.5,5,10,0.770154,2.047588
23,1.0,7,25,0.770177,2.044687
19,1.0,3,25,0.770325,2.025860
22,1.0,7,10,0.770329,2.025416
8,0.3,5,10,0.770359,2.021542
17,0.5,7,25,0.770366,2.020628
